In [ ]:
"""
check_flow_finetune.py
======================
Diagnostic script for the finetuned flow matching model.

Evaluates on the CMIP6 test set (held-out models) and compares against
a training mean baseline (mean dPdP across all training-set models).

For each test model with multiple members, also evaluates per-member
predictions to assess whether the model captures inter-member spread.

Plots
-----
  - Per-example maps: clim, x0, pred, true, residual
  - Scatter: pred vs true (land pixels) for each ensemble member + train mean
  - Per-model RMSE bar chart: flow model vs train mean baseline
  - Per-family RMSE summary
  - Member-level analysis for multi-member test models
  - Training curves (finetune)
  - Ensemble spread calibration

Usage
-----
  python check_flow_finetune.py
"""

import json
import random
from collections import defaultdict
from pathlib import Path

import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch
import xarray as xr

from flow_models import Unet6R
from cmip6_split import (TEST_GROUPS, TRAIN_GROUPS, VAL_GROUPS, ALL_FAMILIES)

# =============================================================================
# configuration
# =============================================================================

CMIP6_DIR      = Path("/Users/ewellmeyer/Documents/research/CMIP6/processed_by_model")
LANDMASK_PATH  = Path("/Users/ewellmeyer/Documents/research/HadGEM/hadgem_landmask_rg128.nc")
WEIGHTS_DIR    = Path("/Users/ewellmeyer/Documents/research/weights")

BASE_CHANNELS  = 16
FINETUNE_EXPT  = (f"flow_finetune_unet6R_ch{BASE_CHANNELS}_oce1.0_aug0.5_noise0.5_zero0.5")
BASE_EXPT      = f"flow_base_unet6R_ch{BASE_CHANNELS}_land10_oce0.3_aug0.8"

FT_DIR   = WEIGHTS_DIR / FINETUNE_EXPT
BASE_DIR = WEIGHTS_DIR / BASE_EXPT
DIAG_DIR = FT_DIR / "diagnostics"
DIAG_DIR.mkdir(exist_ok=True)

N_ENSEMBLE             = 5
ODE_STEPS              = 50
DT                     = 1.0   # integration endpoint: 0 → DT (default 1.0 = full flow)
MEMBER_RMSE_THRESHOLD  = 3.0   # drop members whose mean test RMSE > threshold * median
P_DROP                 = 0.0
N_MAP_EXAMPLES         = 8     # number of test models to show full map diagnostics

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")


# =============================================================================
# helpers
# =============================================================================

def load_group(name, data_dir):
    """Load clim, trend, dPdP for a group. Returns dict or None."""
    clim_path  = data_dir / f"{name}_clim.nc"
    trend_path = data_dir / f"{name}_trend.nc"
    dpdp_path  = data_dir / f"{name}_dPdP.nc"
    if not all(p.exists() for p in [clim_path, trend_path, dpdp_path]):
        return None
    try:
        ds_c = xr.open_dataset(clim_path)
        clim = ds_c["pr_clim"].values.astype(np.float32)
        ds_c.close()

        ds_t = xr.open_dataset(trend_path)
        trend = ds_t["pr_trend"].values.astype(np.float32)
        ds_t.close()
        if trend.ndim == 2:
            trend = trend[np.newaxis, :, :]

        ds_d = xr.open_dataset(dpdp_path)
        dpdp = ds_d["dPdP"].values.astype(np.float32)
        ds_d.close()

        return dict(clim=clim, trends=trend, dpdp=dpdp,
                    n_members=trend.shape[0])
    except Exception as e:
        print(f"  WARN: failed to load {name}: {e}")
        return None


@torch.no_grad()
def integrate(model, x0, clim, steps=20, dt_end=1.0):
    """Euler ODE integration from t=0 to t=dt_end."""
    x = x0.clone()
    dt = dt_end / steps
    for i in range(steps):
        t_val = i * dt
        t = torch.full((x.shape[0],), t_val, device=x.device)
        v = model(x, clim, t)
        x = x + v * dt
    return x


def predict_ensemble(models, x0_np, clim_np, clim_mean, clim_std, dt_end=1.0):
    """
    Run all ensemble members on a single (x0, clim) pair.
    Returns list of predictions (H, W) per member.
    """
    clim_norm = np.nan_to_num((clim_np - clim_mean) / (clim_std + 1e-6))
    x0_clean  = np.nan_to_num(x0_np)

    clim_t = torch.from_numpy(clim_norm[None, None]).float().to(DEVICE)
    x0_t   = torch.from_numpy(x0_clean[None, None]).float().to(DEVICE)

    preds = []
    for _, m in models:
        pred = integrate(m, x0_t, clim_t, steps=ODE_STEPS, dt_end=dt_end)
        preds.append(pred.cpu().numpy()[0, 0])
    return preds


def land_rmse(pred, true, land_flat):
    diff = (pred - true).reshape(-1)[land_flat]
    return np.sqrt(np.mean(diff ** 2))


def land_corr(pred, true, land_flat):
    p = pred.reshape(-1)[land_flat]
    t = true.reshape(-1)[land_flat]
    return np.corrcoef(p, t)[0, 1]


def plot_field(ax, data, title, cmap="RdBu_r", vmin=None, vmax=None,
               symmetric=False):
    if symmetric:
        vmax = vmax or np.nanpercentile(np.abs(data), 98)
        vmin = -vmax
    im = ax.imshow(data, origin="lower", cmap=cmap,
                   vmin=vmin, vmax=vmax, aspect="auto",
                   interpolation="nearest")
    ax.set_title(title, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    return im


# =============================================================================
# load models and data
# =============================================================================

print("Loading clim stats...")
stats_path = FT_DIR / "clim_stats.json"
if not stats_path.exists():
    stats_path = BASE_DIR / "clim_stats.json"
with open(stats_path) as f:
    stats = json.load(f)
clim_mean = stats["clim_mean"]
clim_std  = stats["clim_std"]
print(f"  clim_mean={clim_mean:.2f}  clim_std={clim_std:.2f}")

print("\nLoading finetuned models...")
ft_models = []
for member_idx in range(N_ENSEMBLE):
    ckpt_path = FT_DIR / f"best_member{member_idx}.pth"
    if not ckpt_path.exists():
        print(f"  member {member_idx}: not found, skipping")
        continue
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = Unet6R(input_channels=2, output_channels=1,
                   base_channels=BASE_CHANNELS, p_drop=P_DROP).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    ft_models.append((member_idx, model))
    print(f"  member {member_idx}: epoch {ckpt['epoch']}  "
          f"val={ckpt['val_loss']:.5f}")

# also load base models for comparison
print("\nLoading base models for comparison...")
base_models = []
for member_idx in range(N_ENSEMBLE):
    ckpt_path = BASE_DIR / f"best_member{member_idx}.pth"
    if not ckpt_path.exists():
        continue
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = Unet6R(input_channels=2, output_channels=1,
                   base_channels=BASE_CHANNELS, p_drop=P_DROP).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    base_models.append((member_idx, model))
print(f"  loaded {len(base_models)} base members")

print("\nLoading land mask...")
ds_mask = xr.open_dataset(LANDMASK_PATH)
mask_var = list(ds_mask.data_vars)[0]
land_frac = ds_mask[mask_var].values.astype(np.float32)
land_flat = land_frac.reshape(-1) > 0.5

print("\nLoading test data...")
test_data = {}
for name, family in TEST_GROUPS:
    data = load_group(name, CMIP6_DIR)
    if data is not None:
        data["name"]   = name
        data["family"] = family
        test_data[name] = data
        print(f"  {name:<30} [{family}]  {data['n_members']} members")
    else:
        print(f"  SKIP: {name}")

# load train dPdP for training-mean baseline
print("\nLoading train dPdP for training-mean baseline...")
train_dpdp_list = []

for name, family in TRAIN_GROUPS:
    dpdp_path = CMIP6_DIR / f"{name}_dPdP.nc"
    if dpdp_path.exists():
        try:
            ds = xr.open_dataset(dpdp_path)
            d = ds["dPdP"].values.astype(np.float32)
            ds.close()
            train_dpdp_list.append(d)
        except Exception:
            pass

train_mean_pred = np.mean(train_dpdp_list, axis=0)
print(f"  computed train_mean_pred from {len(train_dpdp_list)} training models")


# =============================================================================
# filter garbage ensemble members
# =============================================================================
# Run each ft member solo across all test models, compute mean RMSE, and drop
# any member whose mean RMSE > MEMBER_RMSE_THRESHOLD * median of all members.

print("\nFiltering ensemble members...")
member_mean_rmse = {}  # member_idx -> mean RMSE across test set

for mi, m in ft_models:
    rmses = []
    for name, info in test_data.items():
        mean_trend = info["trends"].mean(axis=0)
        clim_norm  = np.nan_to_num((info["clim"] - clim_mean) / (clim_std + 1e-6))
        clim_t     = torch.from_numpy(clim_norm[None, None]).float().to(DEVICE)
        x0_t       = torch.from_numpy(np.nan_to_num(mean_trend)[None, None]).float().to(DEVICE)
        pred       = integrate(m, x0_t, clim_t, steps=ODE_STEPS, dt_end=DT)
        pred_np    = pred.cpu().numpy()[0, 0]
        rmses.append(land_rmse(pred_np, info["dpdp"], land_flat))
    member_mean_rmse[mi] = np.mean(rmses)

# print table
print(f"\n  {'MEMBER':>8}  {'MEAN RMSE':>10}  {'STATUS'}")
print(f"  {'-'*35}")
rmse_vals   = np.array(list(member_mean_rmse.values()))
rmse_median = np.median(rmse_vals)
threshold   = MEMBER_RMSE_THRESHOLD * rmse_median
for mi, rmse in sorted(member_mean_rmse.items()):
    status = "DROP (garbage)" if rmse > threshold else "keep"
    print(f"  {mi:>8}  {rmse:>10.4f}  {status}")
print(f"  threshold = {MEMBER_RMSE_THRESHOLD} × median ({rmse_median:.4f}) = {threshold:.4f}")

# plot per-member RMSE bar chart
fig, ax = plt.subplots(figsize=(max(5, len(ft_models) * 1.2), 4))
colors = ["red" if v > threshold else "steelblue"
          for v in member_mean_rmse.values()]
ax.bar([f"m{mi}" for mi in member_mean_rmse],
       member_mean_rmse.values(), color=colors, edgecolor="white")
ax.axhline(threshold, color="red", ls="--", lw=1.5,
           label=f"threshold ({MEMBER_RMSE_THRESHOLD}× median)")
ax.axhline(rmse_median, color="gray", ls=":", lw=1.5,
           label=f"median ({rmse_median:.4f})")
ax.set_ylabel("Mean RMSE across test set (land)")
ax.set_title("Per-member RMSE — red bars will be dropped")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# filter
ft_models_all = ft_models  # keep reference to full list if needed
ft_models     = [(mi, m) for mi, m in ft_models if member_mean_rmse[mi] <= threshold]
dropped       = [mi for mi, _ in ft_models_all if member_mean_rmse[mi] > threshold]
print(f"\n  Dropped members: {dropped}")
print(f"  Keeping {len(ft_models)}/{len(ft_models_all)} members: "
      f"{[mi for mi, _ in ft_models]}")


# =============================================================================
# run inference on test set
# =============================================================================

print("\nRunning inference on test set...")
results = {}

for name, info in test_data.items():
    clim = info["clim"]    # (H, W)
    dpdp = info["dpdp"]    # (H, W)

    # for ensemble-mean evaluation, use member 0's trend (or mean trend)
    mean_trend = info["trends"].mean(axis=0)  # (H, W)

    # finetuned model predictions
    ft_preds = predict_ensemble(ft_models, mean_trend, clim,
                                clim_mean, clim_std, dt_end=DT)
    ft_mean  = np.mean(ft_preds, axis=0)
    ft_std   = np.std(ft_preds, axis=0)

    # base model predictions
    base_preds = predict_ensemble(base_models, mean_trend, clim,
                                  clim_mean, clim_std, dt_end=DT) if base_models else None
    base_mean  = np.mean(base_preds, axis=0) if base_preds else None

    results[name] = dict(
        family          = info["family"],
        n_members       = info["n_members"],
        true            = dpdp,
        clim            = clim,
        trend           = mean_trend,
        ft_preds        = ft_preds,
        ft_mean         = ft_mean,
        ft_std          = ft_std,
        base_mean       = base_mean,
        train_mean_pred = train_mean_pred,
        ft_rmse         = land_rmse(ft_mean, dpdp, land_flat),
        ft_corr         = land_corr(ft_mean, dpdp, land_flat),
        base_rmse       = land_rmse(base_mean, dpdp, land_flat) if base_mean is not None else None,
        base_corr       = land_corr(base_mean, dpdp, land_flat) if base_mean is not None else None,
        train_mean_rmse = land_rmse(train_mean_pred, dpdp, land_flat),
        train_mean_corr = land_corr(train_mean_pred, dpdp, land_flat),
        info            = info,
    )
    print(f"  {name:<30}  ft={results[name]['ft_rmse']:.4f}  "
          f"base={results[name]['base_rmse']:.4f}  "
          f"train_mean={results[name]['train_mean_rmse']:.4f}")


# =============================================================================
# 1. per-model map diagnostics
# =============================================================================

print("\nGenerating map diagnostics...")

map_names = list(test_data.keys())[:N_MAP_EXAMPLES]
for name in map_names:
    r = results[name]
    vdp = np.nanpercentile(np.abs(r["true"]), 97)

    fig, axes = plt.subplots(2, 4, figsize=(20, 7))
    fig.suptitle(f"{name}  [{r['family']}]  "
                 f"RMSE: ft={r['ft_rmse']:.3f}  base={r['base_rmse']:.3f}  "
                 f"train_mean={r['train_mean_rmse']:.3f}",
                 fontsize=10, fontweight="bold")

    plot_field(axes[0, 0], r["clim"], "Clim (mm/yr)", cmap="YlGnBu", vmin=0)
    plot_field(axes[0, 1], r["trend"], "Mean trend (x0)", cmap="BrBG",
               symmetric=True)
    plot_field(axes[0, 2], r["true"], "True dPdP", cmap="BrBG",
               vmin=-vdp, vmax=vdp)
    plot_field(axes[0, 3], r["train_mean_pred"], "Train mean baseline", cmap="BrBG",
               vmin=-vdp, vmax=vdp)
    plot_field(axes[1, 0], r["ft_mean"], "Finetune pred (ens mean)",
               cmap="BrBG", vmin=-vdp, vmax=vdp)
    plot_field(axes[1, 1], r["ft_mean"] - r["true"],
               "Residual (ft - true)", cmap="RdBu_r", symmetric=True)
    plot_field(axes[1, 2], r["ft_std"], "Finetune ens spread (std)",
               cmap="Oranges", vmin=0)
    if r["base_mean"] is not None:
        plot_field(axes[1, 3], r["base_mean"], "Base model pred",
                   cmap="BrBG", vmin=-vdp, vmax=vdp)
    else:
        axes[1, 3].set_visible(False)

    plt.tight_layout()
    # safe = name.replace(" ", "_").replace("/", "-")
    # fig.savefig(DIAG_DIR / f"map_{safe}.png", dpi=120, bbox_inches="tight")
    # plt.close(fig)
    # print(f"  saved map_{safe}.png")
    plt.show()

# =============================================================================
# 2. per-model RMSE bar chart: finetune vs base vs train mean
# =============================================================================

print("\nGenerating RMSE bar chart...")

names_sorted = sorted(results.keys(),
                       key=lambda n: results[n]["ft_rmse"])
n = len(names_sorted)
x = np.arange(n)
w = 0.2

fig, ax = plt.subplots(figsize=(max(12, n * 1.0), 6))
ax.bar(x - w, [results[n]["train_mean_rmse"] for n in names_sorted],
       w, label="Train mean", color="gray", alpha=0.7)
if base_models:
    ax.bar(x,     [results[n]["base_rmse"] for n in names_sorted],
           w, label="Base model", color="red", alpha=0.7)
ax.bar(x + w, [results[n]["ft_rmse"]       for n in names_sorted],
       w, label="Finetuned", color="blue", alpha=0.8)

# mean lines
for label, vals, color, ls in [
    ("Train mean", [results[n]["train_mean_rmse"] for n in names_sorted], "gray",  "--"),
    ("FT",         [results[n]["ft_rmse"]         for n in names_sorted], "blue",  "--"),
]:
    ax.axhline(np.mean(vals), color=color, ls=ls, lw=1.5,
               label=f"{label} mean={np.mean(vals):.3f}")

ax.set_xticks(x)
ax.set_xticklabels([f"{n}\n[{results[n]['family']}]"
                     for n in names_sorted],
                    rotation=45, ha="right", fontsize=7)
ax.set_ylabel("RMSE (land dPdP)")
ax.set_title("Per-model RMSE on test set")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
# fig.savefig(DIAG_DIR / "rmse_per_model.png", dpi=120, bbox_inches="tight")
# plt.close(fig)
# print("  saved rmse_per_model.png")
plt.show()

# =============================================================================
# 3. per-family summary
# =============================================================================

print("\nGenerating per-family summary...")

fam_results = defaultdict(list)
for name, r in results.items():
    fam_results[r["family"]].append(r)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

families = sorted(fam_results.keys())
fam_ft         = [np.mean([r["ft_rmse"]         for r in fam_results[f]]) for f in families]
fam_base       = [np.mean([r["base_rmse"]        for r in fam_results[f]
                            if r["base_rmse"] is not None] or [0])
                  for f in families]
fam_train_mean = [np.mean([r["train_mean_rmse"]  for r in fam_results[f]]) for f in families]

x = np.arange(len(families))
w = 0.2
ax = axes[0]
ax.bar(x - w, fam_train_mean, w, label="Train mean", color="gray",  alpha=0.7)
ax.bar(x,     fam_base,       w, label="Base",        color="red",   alpha=0.7)
ax.bar(x + w, fam_ft,         w, label="Finetuned",   color="blue",  alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(families, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Mean RMSE"); ax.set_title("RMSE by family")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis="y")

# correlation version
fam_ft_r         = [np.mean([r["ft_corr"]         for r in fam_results[f]]) for f in families]
fam_train_mean_r = [np.mean([r["train_mean_corr"]  for r in fam_results[f]]) for f in families]
ax = axes[1]
ax.bar(x - w/2, fam_train_mean_r, w, label="Train mean", color="gray",  alpha=0.7)
ax.bar(x + w/2, fam_ft_r,         w, label="Finetuned",  color="blue",  alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(families, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Mean correlation"); ax.set_title("Pattern correlation by family")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, 1)

plt.tight_layout()
# fig.savefig(DIAG_DIR / "family_summary.png", dpi=120, bbox_inches="tight")
# plt.close(fig)
# print("  saved family_summary.png")
plt.show()

# =============================================================================
# 4. scatter: pred vs true (land pixels, all test models pooled)
# =============================================================================

print("\nGenerating scatter plots...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (label, get_pred, color) in zip(axes, [
    ("Train mean baseline", lambda r: r["train_mean_pred"], "gray"),
    ("Base model",          lambda r: r["base_mean"],       "red"),
    ("Finetuned",           lambda r: r["ft_mean"],         "blue"),
]):
    all_true, all_pred = [], []
    for r in results.values():
        p = get_pred(r)
        if p is None:
            continue
        all_true.append(r["true"].reshape(-1)[land_flat])
        all_pred.append(p.reshape(-1)[land_flat])
    if not all_pred:
        ax.set_visible(False)
        continue

    true_cat = np.concatenate(all_true)
    pred_cat = np.concatenate(all_pred)
    idx = np.random.choice(len(true_cat), size=min(10000, len(true_cat)),
                            replace=False)
    ax.scatter(true_cat[idx], pred_cat[idx], s=1, alpha=0.2, color=color)
    lims = [-20, 20]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_xlabel("True dPdP (land)")
    ax.set_ylabel("Predicted dPdP")
    ax.set_title(label)
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    corr = np.corrcoef(true_cat, pred_cat)[0, 1]
    rmse = np.sqrt(np.mean((pred_cat - true_cat) ** 2))
    ax.text(0.05, 0.95, f"r={corr:.3f}\nRMSE={rmse:.4f}",
            transform=ax.transAxes, va="top", fontsize=10,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

plt.suptitle("Pred vs True dPdP — test set (land pixels)", fontsize=11)
plt.tight_layout()
# fig.savefig(DIAG_DIR / "scatter_all.png", dpi=120, bbox_inches="tight")
# plt.close(fig)
# print("  saved scatter_all.png")
plt.show()

# =============================================================================
# 5. member-level analysis for multi-member test models
# =============================================================================

print("\nGenerating member-level analysis...")

multi_member = {name: r for name, r in results.items()
                if r["info"]["n_members"] >= 2}

if multi_member:
    for name, r in multi_member.items():
        info    = r["info"]
        n_mem   = info["n_members"]
        clim    = info["clim"]
        dpdp    = info["dpdp"]
        trends  = info["trends"]  # (n_members, H, W)

        # predict for each member's trend independently
        member_preds = []
        for m_idx in range(n_mem):
            preds = predict_ensemble(ft_models, trends[m_idx], clim,
                                     clim_mean, clim_std, dt_end=DT)
            member_preds.append(np.mean(preds, axis=0))  # ens mean over seeds

        member_preds = np.stack(member_preds, axis=0)  # (n_mem, H, W)
        pred_spread  = member_preds.std(axis=0)
        pred_mean    = member_preds.mean(axis=0)

        # per-member RMSE
        member_rmse = [land_rmse(member_preds[i], dpdp, land_flat)
                       for i in range(n_mem)]

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.suptitle(f"{name} [{r['family']}] — {n_mem} members",
                     fontsize=10, fontweight="bold")

        # member RMSE distribution
        ax = axes[0]
        ax.hist(member_rmse, bins=min(20, n_mem // 2 + 1),
                color="blue", alpha=0.7, edgecolor="white")
        ax.axvline(r["train_mean_rmse"], color="gray", ls="--", lw=2,
                   label=f"Train mean={r['train_mean_rmse']:.3f}")
        ax.axvline(np.mean(member_rmse), color="blue", ls="--", lw=2,
                   label=f"mean={np.mean(member_rmse):.3f}")
        ax.set_xlabel("RMSE per member trend")
        ax.set_ylabel("Count")
        ax.set_title("Member-level RMSE distribution")
        ax.legend(fontsize=8)

        # prediction spread vs true pattern
        vdp = np.nanpercentile(np.abs(dpdp), 97)
        plot_field(axes[1], pred_mean, "Mean pred across member trends",
                   cmap="BrBG", vmin=-vdp, vmax=vdp)
        plot_field(axes[2], pred_spread, "Pred spread (std across member trends)",
                   cmap="Oranges", vmin=0)

        plt.tight_layout()
        # safe = name.replace(" ", "_").replace("/", "-")
        # fig.savefig(DIAG_DIR / f"members_{safe}.png", dpi=120,
        #             bbox_inches="tight")
        # plt.close(fig)
        # print(f"  saved members_{safe}.png")
        plt.show()
else:
    print("  no multi-member test models with >= 5 members")


# =============================================================================
# 6. ensemble spread calibration
# =============================================================================

print("\nGenerating ensemble spread calibration...")

all_spread = []
all_error  = []
for r in results.values():
    spread = r["ft_std"].reshape(-1)[land_flat]
    error  = np.abs(r["ft_mean"] - r["true"]).reshape(-1)[land_flat]
    all_spread.append(spread)
    all_error.append(error)

spread_cat = np.concatenate(all_spread)
error_cat  = np.concatenate(all_error)

# bin by spread and compute mean error per bin
n_bins = 100
percentiles = np.linspace(0, 100, n_bins + 1)
bin_edges   = np.percentile(spread_cat, percentiles)
bin_spread  = []
bin_error   = []
for i in range(n_bins):
    mask = (spread_cat >= bin_edges[i]) & (spread_cat < bin_edges[i+1])
    if mask.sum() > 0:
        bin_spread.append(spread_cat[mask].mean())
        bin_error.append(error_cat[mask].mean())

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(bin_spread, bin_error, s=40, color="steelblue", zorder=5)
lims = [0, max(max(bin_spread), max(bin_error)) * 1.1]
ax.plot(lims, lims, "k--", lw=1, label="perfect calibration")
ax.set_xlabel("Ensemble spread (std of predictions)")
ax.set_ylabel("Mean absolute error")
ax.set_title("Spread-error calibration (land pixels, test set)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
# fig.savefig(DIAG_DIR / "spread_calibration.png", dpi=120, bbox_inches="tight")
# plt.close(fig)
plt.show()
print("  saved spread_calibration.png")


# =============================================================================
# 7. training curves
# =============================================================================

print("\nGenerating training curves...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for member_idx, _ in ft_models:
    log_path = FT_DIR / f"log_member{member_idx}.json"
    if not log_path.exists():
        continue
    with open(log_path) as f:
        log = json.load(f)
    epochs = [e["epoch"] for e in log]
    train  = [e["train"] for e in log]
    val    = [e["val"]   for e in log]
    lr     = [e.get("lr", 0) for e in log]

    axes[0].plot(epochs, train, lw=1.5, label=f"m{member_idx} train")
    axes[0].plot(epochs, val,   lw=1.5, ls="--", label=f"m{member_idx} val")
    axes[1].plot(epochs, lr,    lw=1.5, label=f"m{member_idx}")

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Masked MSE")
axes[0].set_title("Finetune training curves")
axes[0].set_yscale("log")
axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Learning rate")
axes[1].set_title("LR schedule")
axes[1].set_yscale("log")
axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
# fig.savefig(DIAG_DIR / "training_curves.png", dpi=120, bbox_inches="tight")
# plt.close(fig)
plt.show()
print("  saved training_curves.png")


# =============================================================================
# 8. print summary table
# =============================================================================

print(f"\n{'='*80}")
print(f"{'MODEL':<30} {'FAMILY':<12} {'#MEM':>5}  "
      f"{'FT':>7} {'BASE':>7} {'TRAIN_M':>7}  "
      f"{'FT>TM?':>7}")
print(f"{'='*80}")

ft_wins = 0
for name in sorted(results.keys()):
    r = results[name]
    beat = "YES" if r["ft_rmse"] < r["train_mean_rmse"] else "no"
    if beat == "YES":
        ft_wins += 1
    base_str = f"{r['base_rmse']:.4f}" if r["base_rmse"] is not None else "  n/a  "
    print(f"  {name:<28} {r['family']:<12} {r['n_members']:>5}  "
          f"{r['ft_rmse']:>7.4f} {base_str} {r['train_mean_rmse']:>7.4f}  "
          f"{beat:>7}")

n_test = len(results)
mean_ft         = np.mean([r["ft_rmse"]         for r in results.values()])
mean_base       = np.mean([r["base_rmse"]        for r in results.values()
                            if r["base_rmse"] is not None])
mean_train_mean = np.mean([r["train_mean_rmse"]  for r in results.values()])

print(f"\n{'MEAN':<30} {'':12} {'':>5}  "
      f"{mean_ft:>7.4f} {mean_base:>7.4f} {mean_train_mean:>7.4f}  "
      f"{ft_wins}/{n_test} beat train mean")

print(f"\nAll diagnostics saved to {DIAG_DIR}")

Loading clim stats...
  clim_mean=895.57  clim_std=841.56

Loading finetuned models...
  member 0: epoch 23  val=91.76763
  member 1: epoch 23  val=68.84050
  member 2: epoch 21  val=78.21535
  member 3: epoch 19  val=77.29130
  member 4: epoch 30  val=90.24674

Loading base models for comparison...
